# Customer Support Chatbot — Student Notebook

## 🎯 Session Goal
Build a **Retrieval-Augmented Generation (RAG)** FAQ chatbot in 7 steps:
1. Load 500-row FAQ dataset from CSV
2. Chunk & embed with OpenAI
3. Store vectors in Chroma (persistent database)
4. Build a RAG chain (retriever → prompt → LLM)
5. Test with real questions

## 📋 Prerequisites
- `.env` file with `OPENAI_API_KEY` set
- Run `python data/generate_faq.py` once to create the FAQ CSV

## ⚡ Live Coding Session
Follow each step below. **Do not skip cells.** Run top-to-bottom once.

---

## Step 1: Imports & Setup

**What we'll do:**
- Import LangChain tools for LLMs, embeddings, text splitting, and vector stores
- Import Pandas for data loading
- Load environment variables (API keys) from `.env`
- Initialize the embedder (OpenAI text-embedding-3-small) and LLM (GPT-4o-mini)

**Key concepts:**
- **Embedder**: Converts text → vectors for semantic search
- **LLM**: The language model answering questions
- **RecursiveCharacterTextSplitter**: Chunks long docs smartly
- **Chroma**: Persistent vector store (survives kernel restarts)

**Your task:** Write code to import all libraries and initialize the LLM + embedder.

In [ ]:
# ============================================================================
# STEP 1: Imports & Setup
# ============================================================================
# This cell initializes all required libraries and loads your API keys from .env
# - We import os, dotenv to manage environment variables securely (no hardcoded keys!)
# - We import pandas to load the FAQ dataset from CSV
# - We load OPENAI_API_KEY from environment (will prompt if not found)
# - We also initialize LangSmith for tracing (debugging & monitoring tool)
# ============================================================================

import os
from dotenv import load_dotenv
import pandas as pd
from getpass import getpass

# Load .env if present
load_dotenv()

# Load the Openai+langsmith skeys:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass("OPENAI_API_KEY:")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGSMITH_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT", "CUSTOMER_SUPPORT_PROJECT")
os.environ['LANGCHAIN_TRACING_V2'] = "true"

print("LangSmith Tracing is ON for this Notebook")


LangSmith Tracing is ON for this Notebook


In [ ]:
# ============================================================================
# Import LangChain Components for RAG + Memory
# ============================================================================

# LLM & Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # GPT-4o-mini for chat, text-embedding-3-small for vectors

# Text Processing
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smartly chunks long documents

# Vector Store
from langchain_community.vectorstores import Chroma  # Persistent vector database

# Document Wrapper
from langchain_core.documents import Document  # Wrapper class that holds text content + metadata (source, tags, etc.,)

# Prompts & Output
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # ChatPromptTemplate - Format prompts with variables # MessagesPlaceholder = ALoows dynamic insersation of conversation history
from langchain_core.output_parsers import StrOutputParser  # Extract text from LLM output responses

# Runnable Components (Chains)
from langchain_core.runnables import RunnablePassthrough  # Passes the inputs unchanged

# Memory & Conversation
from langchain_core.runnables.history import RunnableWithMessageHistory  # Manages converation memory by storing and retriving the chat history per session
from langchain_core.chat_history import InMemoryChatMessageHistory  # Stores chat messages in memory

# ============================================================================


In [ ]:
# ============================================================================
# Initialize the Embedder & LLM
# ============================================================================
# Embedder: Converts text (questions, FAQ) into vectors (arrays of numbers)
#   - Used for semantic similarity search: "reset password" is similar to "forgot password"
#   - Model: text-embedding-3-small (fast, cheap, good quality)
#
# LLM: The language model that generates responses
#   - Model: GPT-4o-mini (fast, cheap, good quality)
#   - temperature=0: Deterministic (same input = same output, best for support)
# ============================================================================

embedder = OpenAIEmbeddings(model = "text-embedding-3-small")

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

print("Step 1: Ready to code imports & setup")


Step 1: Ready to code imports & setup


---

## Step 2: Verify API Keys

**What we'll do:**
- Assert that `OPENAI_API_KEY` is set (either in `.env` or as environment variable)
- Print a success message if the key is found

**Why:** No key = no LLM calls. Fail fast!

**Your task:** Write an assertion to check the key exists.

In [ ]:
# ============================================================================
# STEP 2: Verify API Keys
# ============================================================================
# Safety check: Fail FAST if API key is missing (avoids wasting time on errors)
# This ensures the notebook can call OpenAI's LLM and embeddings
# ============================================================================

# STEP 2: Check that OPENAI_API_KEY exists
# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY!"
# print("✅ API key loaded.")

print("Step 2: Ready to verify keys")


Step 2: Ready to verify keys


---

## Step 3: Load the FAQ Dataset

**What we'll do:**
- Load `data/customer_support_faq.csv` using Pandas
- Convert each row (question + answer) into a LangChain `Document` object
- Keep metadata: region, channel, issue (for filtering later)

**Key concept:**
- **Document**: A container with `page_content` (text) and `metadata` (tags)
- Each FAQ row becomes one Document object

**Your task:** Load CSV, convert to Documents with metadata.

In [ ]:
# ============================================================================
# STEP 3: Load the FAQ Dataset
# ============================================================================
# This cell loads the FAQ CSV file (500 Q&A pairs) and converts each row
# into a Python dictionary for easy iteration.
#
# What happens:
# 1. Read CSV file with Pandas (columns: question, answer, region, channel, issue)
# 2. Convert DataFrame rows → list of dictionaries (easier to loop)
# 3. Each dict has keys: question, answer, region, channel, issue
#
# Why dictionaries? Easy access: row['question'], row['answer'], etc.
# ============================================================================

import pandas as pd
from pathlib import Path  # Cross-platform file path handling

#csv_path = Path("data/customer_support_faq.csv")

faq_df = pd.read_csv("data/customer_support_faq.csv")

faq_items = faq_df.to_dict(orient = "records")  # Convert the Dataframe rows to list of dicts for much easier iterations


In [ ]:
# ============================================================================
# Convert FAQ Rows → LangChain Document Objects
# ============================================================================
# LangChain uses Document objects to manage content + metadata together.
#
# What we're doing:
# 1. Create a Document for each FAQ pair (Q + A)
# 2. page_content = formatted question + answer text
# 3. metadata = source info (region, channel, issue) for later filtering
#
# Why Document objects?
# - LangChain's retriever expects Document objects
# - Metadata allows filtering results by region, channel, etc.
# - Each document is a self-contained unit ready for embedding
#
# Example document:
# Document(
#   page_content="Q: How do I reset my password?\nA: Click forgot password...",
#   metadata={"source": "faq_csv", "region": "US", "channel": "web", "issue": "account"}
# )
# ============================================================================

from langchain_core.documents import Document 

documents = [
    Document(
    
    page_content=f"Q:{row['question']}\nA:{row['answer']}",
    
    metadata={"source":"faq_csv", "row": idx, "region": row.get("region"),
              "chennel":row.get("channel"), "issue": row.get("issue")}
)
for idx, row in faq_df.iterrows()  # Iterate through each FAQ rows
]

len(documents)  # Should print: 500 (total FAQ pairs)


500

---

## Step 4: Build a Persistent Vector Store (Chroma)

**What we'll do:**
- Embed all chunks using the OpenAI embedder
- Store embeddings + text in **Chroma** at `data/chroma_faq/`
- Create a `retriever` to fetch relevant chunks for a query

**Key concepts:**
- **Chroma**: Persistent vector database—embeddings survive kernel restarts!
- **Retriever**: Finds the K most similar chunks to a user query (semantic search)

**Demo after coding:** Test with `retriever.get_relevant_documents("password reset")` to see top matches.

### No Chunking:



In [ ]:
# ============================================================================
# STEP 4: Build Persistent Vector Store (Chroma)
# ============================================================================
# This cell embeds all FAQ documents and stores them in Chroma (persistent database).
#
# What happens:
# 1. Each Document is embedded using OpenAI embedder (text → 1536-dim vector)
# 2. Vectors + text stored in Chroma at data/chroma_faq/
# 3. Persist directory = survives kernel restarts (your data is SAFE)
# 4. Retriever interface created for semantic search
#
# Why Chroma?
# - Persistent: embeddings saved to disk (no re-embedding needed)
# - Fast: semantic search in milliseconds
# - Easy: simple API, great for RAG pipelines
#
# Next step: When user asks "how do I reset password?":
# 1. Embed their question
# 2. Find most similar FAQ vectors (cosine similarity)
# 3. Return top-K matching FAQs as context for LLM
# ============================================================================

# STEP 5: Build Chroma vector store

persist_dir = "data/chroma_faq"

vectorstore = Chroma.from_documents(
    documents = documents,  # User Original Complete Q & A Pairs (Not Chunks)
    embedding = embedder,
    persist_directory=persist_dir
)

retriever = vectorstore.as_retriever()  # Interface to query and the vector store

print(f"✅ Chroma vector store ready at {persist_dir}")


✅ Chroma vector store ready at data/chroma_faq


# Session-Based Conversation Memory

## Why Session Memory?

In customer support, each customer has their own conversation history:
- **Session ID** = unique identifier (e.g., "user_session_100", "customer_12345")
- **Persistent store** = dictionary mapping session_id → chat history
- **Isolation** = No cross-talk between customers (Alice doesn't see Bob's conversation)

## How It Works:

1. Customer initiates chat → New session_id created
2. `store` dictionary stores their conversation history
3. `get_session_history()` retrieves it on next message
4. LLM always has access to past messages for context

**Like a support ticket system:**
- Ticket #100 has its own history
- Support agent reads full history when they pick up the ticket
- Each new message appended to that ticket's history


In [ ]:
# ============================================================================
# STEP 5: Build RAG Chain with Memory
# ============================================================================
# This cell builds the complete RAG pipeline with conversation memory.
# 
# Memory Store:
# - store = {} is a memory bank, each customer has isolated conversation history
# - get_session_history() ensures new customers get new memory, existing customers continue
# ============================================================================

store = {}  # Dictionary to track per-session chat histories : {session_id - > InMemoryChatMessageHistory}

# New Customer -> New Memory
# Existing Customer -> Continue the Conversation
def get_session_history(session_id: str):
    """Get or Create chathistory for a session"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ============================================================================
# RAG Prompt Template
# ============================================================================
# This template instructs the LLM how to behave and where to use the context.
# Placeholders:
# - {context} = retrieved FAQ chunks (filled by retriever)
# - {question} = current user question
# - {history} = past conversation messages (filled by memory system)
# ============================================================================

template = """You are a helpful customer support chatbot.
Use the following context to answer the question.
If you don't know, say you don't know.

Context: {context}  # Retrieved FAQ chunks inserted here

Question: {question}  # User's current question

Answer:"""  # LLM generated response based on context + history


prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    MessagesPlaceholder(variable_name = "history"),  ## injects past conversations
    ("human", "{question}")
])


# ============================================================================
# Format Documents for LLM
# ============================================================================
# The retriever returns Document objects (with page_content + metadata)
# The LLM expects plain text. This function converts:
# Document(page_content="Q: ...\nA: ...", metadata={...})
# → "Q: ...\nA: ..."
# ============================================================================

# This function transformes retriever output into LLM-ready text.
# Taks a list of document objects and converts them into a single readable string.
# The retriever returns Documents Objects
# LLM understands plain text
# 

def format_docs(docs):
    """Convert Document objects to plain text for LLM"""
    return "\n\n".join(doc.page_content for doc in docs)

# ============================================================================
# Build the RAG Chain (without memory)
# ============================================================================
# RunnableParallel executes multiple tasks simultaneously:
# 1. context = retrieve top FAQ chunks for the question
# 2. question = pass through the user's question
# 3. history = will be injected by RunnableWithMessageHistory later
# Then pipe (|) to:
#   prompt → format everything into the template
#   llm → call GPT-4o-mini
#   StrOutputParser → extract text from response
# ============================================================================

from langchain_core.runnables import RunnableParallel
# Orchestrator Engine - 
# 1. Runs multiple operations at the same time
# 2. Combines results into one dictionary

# RunnableParallel: Executes all 3 tasks simultaneously, combines results into single dict
# Prepare 3 pieces

base_chain = (
    RunnableParallel( 
    context = lambda x: format_docs(retriever.invoke(x.get("question", ""))),  # Retrieve FAQ context
    question = lambda x: x.get("question", ""),  # Pass through question
    history = lambda x: x.get("history", [])  # History will be injected later
)
| prompt  # Format into template
| llm  # Call LLM
| StrOutputParser()  # Extract text output
)

# ============================================================================
# Add Memory Management to Chain
# ============================================================================
# RunnableWithMessageHistory wraps the base_chain to:
# 1. Load session history before each call
# 2. Pass history to the LLM (for context awareness)
# 3. Save new messages to history after each call
# ============================================================================

# Saves every user message
# Saves every AI response
# Reloads history on next turn

qa_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,  # Function to get/create session memory
    input_messages_key= "question",  # User input field
    history_messages_key = "history"  # Where to inject past messages
)

# ============================================================================


#### User Question -> Load Session History -> Retrieve Relevant FAQs -> Build RAG Prompt -> LLM Generates Answer -> Save Conversation to Memory

In [ ]:
# ============================================================================
# STEP 6: Test the Chatbot with Memory
# ============================================================================
# This cell demonstrates multi-turn conversation with memory.
#
# What happens:
# 1. User asks question #1 → Retrieve FAQ context → LLM answers → Save Q&A to memory
# 2. User asks question #2 → LLM now has BOTH question #1 & #2 in context
# 3. User asks question #3 → Full conversation history available to LLM
#
# This allows the LLM to:
# - Understand follow-up questions ("it" refers to previous answer)
# - Maintain consistent support experience
# - Reference earlier issues in conversation
# ============================================================================

session_id = "user_session_100"

## helper funtion to chat with memory
def chat_with_memory(user_input, session_id = session_id):
    """
    Ask a question with conversation history, 
    1. pass the question to q_chain
    2. Chain retrieves FAQ context + loads history for session_id
    3. LLM answers using context+history
    4. New Q&A automatically saved to session history
    5. All steps traced in LangSmith
    """
    config = {"configurable": {"session_id": session_id}}  # Tell RunnableWithMessage History which session to use
    answer = qa_chain.invoke({"question": user_input}, config = config)  # Loads past conversations, Retrieves relevent FAQs, Build RAG Prompt, Calls LLMs
    return answer

# Test with multiple questions to demonstrate memory
questions = [
      "How do I handle reset password via the web app?",
      "How do I handle update payment method via the web app?",
      "How do I handle two-factor auth via the mobile app?"
]

for i, q in enumerate(questions, 1):
    answer = chat_with_memory(q)
    print(f"Turn {i}")
    print(f"Customer: {q}")
    print(f"Support Bot: {answer}\n")


Tuen 1
Customer: How do I handle reset password via the web app?
Support Bot: For reset password on the web app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.

Tuen 2
Customer: How do I handle update payment method via the web app?
Support Bot: For update payment method on the web app, visit Billing > Payments and refresh your card on file. If it persists, open a ticket with logs, region=EU, and a short description. Typical resolution: 24h.

Tuen 3
Customer: How do I handle two-factor auth via the mobile app?
Support Bot: For two-factor auth on the mobile app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.



User Question -> Session ID -> Load Conversation History -> Retrieve Relevant FAQs -> Build RAG PRompt -> LLM Answer -> Save History -> Trace Every Thing

---

## Step 6: Build the RAG Chain

**What we'll do:**
- Create a **Prompt Template** with placeholders: `{context}` and `{question}`
- Wire together:
  1. **Retrieval**: Get relevant chunks for the user's question
  2. **Prompt**: Fill in context + question
  3. **LLM**: Generate an answer
  4. **Parse**: Extract the text response

**RAG pipeline flowchart:**
```
User Question
    ↓
[Retriever] → Find top-K relevant chunks
    ↓
[Prompt Template] → Format: "You are a support bot. Context: {chunks}. Q: {q}. Answer:"
    ↓
[LLM (GPT-4o-mini)] → Generate response
    ↓
[StrOutputParser] → Return plain text
```

**Key point:** `RunnableParallel` fetches context and question in parallel before feeding to LLM.

In [ ]:
# ============================================================================
# STEP 7: Test the Chatbot with Real Questions
# ============================================================================
# This is a commented template showing how to build a basic RAG chain
# (without memory). The full implementation with memory is above.
#
# Key components:
# 1. rag_chain = Retriever → Prompt → LLM → Output Parser
# 2. Useful for simple Q&A without conversation history
# 3. More efficient for one-off questions
#
# For production: Use the qa_chain above (includes memory)
# ============================================================================

# STEP 7: Build the RAG chain
# template = """You are a helpful customer support chatbot.
# Use the context to answer the question.
# If you don't know, say you don't know.
#
# Context: {context}
# Question: {question}
# Answer:"""
#
# prompt = ChatPromptTemplate.from_template(template)
#
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)
#
# rag_chain = (
#     RunnableParallel(
#         context=lambda x: format_docs(retriever.invoke(x.get("question", ""))),
#         question=lambda x: x.get("question", "")
#     )
#     | prompt
#     | llm
#     | StrOutputParser()
# )
#
# print("✅ RAG chain ready!")

print("Step 7: Ready to test the chatbot with real questions")


---

## Step 7: Test with Real Questions

**What we'll do:**
- Ask the chain a question like: _"How do I reset my password via the mobile app?"_
- Watch it retrieve relevant FAQ chunks and generate an answer

**What happens under the hood:**
1. User question is embedded
2. Retriever finds top-K matching chunks from Chroma
3. Chunks are formatted into the prompt
4. LLM reads prompt + context and generates a response

**Try these questions:**
- "How do I reset my password via the mobile app?"
- "What is your refund policy?"
- "Do you offer phone support?"
- (Make up your own!)

In [ ]:
# ============================================================================
# Example: Simple RAG Chain (no memory) — Template Reference
# ============================================================================
# This shows how to test a basic RAG chain without conversation memory.
# Useful for:
# - One-off Q&A (no context needed from previous turns)
# - Simple FAQ lookup
# - Testing retrieval without memory overhead
#
# How it works:
# question → embed → retrieve FAQs → build prompt → LLM → answer
#
# Production use: Use qa_chain (with memory) instead
# ============================================================================

# STEP 7: Test the chain with questions
# questions = [
#     "How do I reset my password via the mobile app?",
#     "What is your refund policy?",
#     "Do you offer phone support?"
# ]
#
# for q in questions:
#     answer = rag_chain.invoke({"question": q})
#     print(f"Q: {q}")
#     print(f"A: {answer}\n")

print("Step 7: Ready to test the chatbot")


---

## Advanced: Memory Strategies (Steps 8-10)

### Why Different Memory Strategies?

Long conversations get expensive—every API call includes the full history. Here are three approaches to manage conversation context:

1. **Buffer Memory**: Keep ALL messages (full context, highest token cost)
2. **Window Memory**: Keep only LAST N messages (balanced, fixed cost)
3. **Summary Memory**: Summarize old messages with LLM (lowest cost, may lose detail)

Each has trade-offs. Choose based on your use case!

---

## Step 8: Buffer Memory (All Messages)

**What we'll do:**
- Initialize a separate in-memory store for this strategy
- Keep ALL conversation messages
- Show how memory size grows with each turn
- Perfect for short, focused conversations

**Key concept:**
- **Buffer**: No message loss, but token usage scales linearly
- **Use case**: Customer support chats (typically 3-5 turns)

**Your task:** Implement buffer memory storage and chat loop.

In [ ]:
# ============================================================================
# APPROACH 1: Buffer Memory (Keep ALL Messages)
# ============================================================================
# Buffer memory stores every single message ever sent.
#
# Trade-offs:
# ✅ Pros:
#    - No message loss, full context available
#    - Best for understanding complete conversation flow
#    - Highest accuracy (LLM sees all context)
#
# ❌ Cons:
#    - Token usage grows linearly with conversation length
#    - Expensive for long conversations (100 turns = massive prompt)
#    - Best for short, focused chats (3-5 turns typical)
#
# Use case: Customer support tickets (usually 3-5 turns per issue)
# ============================================================================

# Reset for clean demo
store_buffer = {}  # Separate store from earlier example

def get_session_history_buffer(session_id: str):
    """Buffer memory: no deletion, stores all messages"""
    if session_id not in store_buffer:
        store_buffer[session_id] = InMemoryChatMessageHistory()  # Fresh history per session
    return store_buffer[session_id]

qa_chain_buffer = RunnableWithMessageHistory(
    base_chain,
    get_session_history_buffer,  # Use buffer-specific history getter
    input_messages_key="question",
    history_messages_key="history"
)

def chat_buffer(user_input, session_id="buffer_session"):
    """Chat using buffer memory (retains all previous messages)"""
    config = {"configurable": {"session_id": session_id}}
    return qa_chain_buffer.invoke({"question": user_input}, config=config)  # All messages sent to LLM each turn

print("=== BUFFER MEMORY (All Messages) ===\n")
for i, q in enumerate(questions, 1):
    answer = chat_buffer(q)  # Memory grows with each turn
    history = store_buffer["buffer_session"].messages  # Retrieve all stored messages
    print(f"Turn {i}: {q}")

    print(f"Bot: {answer}")
    print(f"Memory size: {len(history)} messages\n")  # Should grow: 2 -> 4 -> 6 messages


In [ ]:
# ============================================================================
# Buffer Memory Implementation Reference (Template)
# ============================================================================
# This shows how you could manually implement buffer memory if you wanted to
# control the logic yourself (without LangChain's RunnableWithMessageHistory).
#
# Steps:
# 1. Create separate store for buffer strategy
# 2. Get or create history for session_id
# 3. After each response, optionally trim history (but we DON'T for buffer)
# 4. Track growing memory size
#
# Note: The actual working implementation is in the cell above using
# RunnableWithMessageHistory (more robust and production-ready).
# ============================================================================

# STEP 8: Buffer Memory (Keep ALL messages)
# store_buffer = {}
#
# def get_session_history_buffer(session_id: str):
#     if session_id not in store_buffer:
#         store_buffer[session_id] = InMemoryChatMessageHistory()
#     return store_buffer[session_id]
#
# qa_chain_buffer = RunnableWithMessageHistory(
#     base_chain,
#     get_session_history_buffer,
#     input_messages_key="question",
#     history_messages_key="history"
# )
#
# def chat_buffer(user_input, session_id="buffer_session"):
#     config = {"configurable": {"session_id": session_id}}
#     return qa_chain_buffer.invoke({"question": user_input}, config=config)
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_buffer(q)
#     history_size = len(store_buffer["buffer_session"].messages)
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     print(f"Memory: {history_size} messages\n")

print("Step 8: Ready to implement buffer memory")


---

## Step 9: Window Memory (Last N Messages)

**What we'll do:**
- Keep only the LAST N messages (e.g., last 4 messages = 2 conversation turns)
- Discard older context automatically
- Show fixed token cost regardless of conversation length

**Key concepts:**
- **Window size**: How many recent messages to keep (e.g., 4 = last 2 turns)
- **Trade-off**: Loses older context but cost is predictable
- **Use case**: Long chatbots where you only need recent context

**Why 4 messages?** 1 turn = 1 user + 1 assistant = 2 messages. Window of 4 = keep last 2 turns.

**Your task:** Implement window-based history trimming.

In [ ]:
# ============================================================================
# APPROACH 2: Window Memory (Last N Messages Only)
# ============================================================================
# Window memory keeps only the most recent N messages (fixed context window).
#
# Trade-offs:
# ✅ Pros:
#    - Predictable token cost (always max_tokens messages)
#    - Good balance between cost and context
#    - Works for longer conversations (10-20 turns)
#
# ❌ Cons:
#    - Loses older context (customer can't reference turn 1 in turn 10)
#    - May miss important information from earlier turns
#    - Requires tuning window size
#
# Config:
# - window_size = 4 means keep last 4 messages = 2 complete turns
# - Adjust based on your conversation patterns
#
# Use case: Multi-turn chatbots (FAQ, research, document Q&A)
# ============================================================================

from langchain_core.messages import trim_messages  # LangChain's built-in message trimming utility

store_window = {}  # Separate store for window memory demo
window_size = 4  # Keep last 4 messages = 2 complete turns (2 human + 2 AI)

def get_session_history_window(session_id: str):
    """Window memory: get or create history, LangChain handles trimming"""
    if session_id not in store_window:
        store_window[session_id] = InMemoryChatMessageHistory()  # Fresh history per session
    
    # Use LangChain's trim_messages to automatically keep only last N messages
    history = store_window[session_id]
    trimmed_messages = trim_messages(
        history.messages,
        max_tokens=window_size,  # Keep last 4 messages
        strategy="last",  # Keep most recent messages
        token_counter=len,  # Simple count: 1 message = 1 token
    )
    
    # Update history with trimmed messages
    history.messages = trimmed_messages
    return history

# Create separate chain for window memory
qa_chain_window = RunnableWithMessageHistory(
    base_chain,
    get_session_history_window,  # Use window-specific history getter with auto-trim
    input_messages_key="question",
    history_messages_key="history"
)

def chat_window(user_input, session_id="window_session"):
    """Chat using sliding window memory (LangChain auto-trims old messages)"""
    config = {"configurable": {"session_id": session_id}}
    # LangChain's trim_messages automatically called in get_session_history_window
    answer = qa_chain_window.invoke({"question": user_input}, config=config)
    return answer

print("=== WINDOW MEMORY (Last 4 Messages Only) ===\n")
for i, q in enumerate(questions, 1):
    answer = chat_window(q)  # Memory auto-trimmed by LangChain
    history = store_window["window_session"].messages  # Retrieve trimmed history
    print(f"Turn {i}: {q}")
    print(f"Bot: {answer}")
    print(f"Memory size: {len(history)} messages (window_size={window_size})")
    
    # Show which messages are currently in the window
    print("Messages in window:")
    for idx, msg in enumerate(history, 1):
        msg_type = "Human" if hasattr(msg, 'type') and msg.type == "human" else "AI"
        preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
        print(f"  [{idx}] {msg_type}: {preview}")
    print()  # Stays at 4 after first 2 turns


=== WINDOW MEMORY (Last 4 Messages Only) ===

Turn 1: How do I handle reset password via the web app?
Bot: For reset password on the web app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.
Memory size: 2 messages (window_size=4)
Messages in window:
  [1] Human: How do I handle reset password via the web app?
  [2] AI: For reset password on the web app, open Settings > Security ...

Turn 2: How do I handle update payment method via the web app?
Bot: For update payment method on the web app, visit Billing > Payments and refresh your card on file. If it persists, open a ticket with logs, region=EU, and a short description. Typical resolution: 24h.
Memory size: 4 messages (window_size=4)
Messages in window:
  [1] Human: How do I handle reset password via the web app?
  [2] AI: For reset password on the web app, open Settings > Security ...
  [3] Human: How do I handle update payment meth

In [ ]:
# ============================================================================
# Window Memory Implementation Reference (Template)
# ============================================================================
# This shows how you could manually implement window memory with custom
# message trimming logic.
#
# Steps:
# 1. Create trim_window_history() function
# 2. Keep only last N messages manually
# 3. Track memory size (should stay constant after initial fill)
#
# Note: The actual working implementation uses LangChain's trim_messages()
# (in the cell above) which is more robust.
# ============================================================================

# STEP 9: Window Memory (Keep last N messages only)
# from langchain_core.messages import HumanMessage, AIMessage
#
# store_window = {}
# window_size = 4  # Keep last 4 messages (2 turns)
#
# def get_session_history_window(session_id: str):
#     if session_id not in store_window:
#         store_window[session_id] = InMemoryChatMessageHistory()
#     return store_window[session_id]
#
# def trim_window_history(history_obj, window_size=4):
#     """Keep only last N messages"""
#     messages = history_obj.messages
#     if len(messages) > window_size:
#         return messages[-window_size:]
#     return messages
#
# # Custom chain wrapper for window memory
# def chat_window(user_input, session_id="window_session"):
#     history_obj = get_session_history_window(session_id)
#     config = {"configurable": {"session_id": session_id}}
#     answer = qa_chain.invoke({"question": user_input}, config=config)
#     
#     # Trim history to window size
#     trimmed = trim_window_history(history_obj, window_size)
#     history_obj.messages = trimmed
#     return answer
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_window(q)
#     history_size = len(store_window["window_session"].messages)
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     print(f"Memory: {history_size}/{window_size} messages\n")

print("Step 9: Ready to implement window memory")


---

## Step 10: Summary Memory (LLM-Based Compression)

**What we'll do:**
- Use the LLM itself to SUMMARIZE old messages
- Keep recent N turns unmodified (full detail)
- Replace older turns with 1-2 sentence summaries
- Most efficient for very long conversations

**Key concepts:**
- **keep_recent_messages**: How many recent turns to preserve fully (e.g., 2)
- **Summarization**: Use LLM to compress old turns into key points
- **Use case**: Research sessions, long document Q&A, multi-hour support chats

**Trade-off:** Cheapest but loses granular detail from old context.

**Your task:** Implement LLM-based message compression.

In [ ]:
# ============================================================================
# Summary Memory Implementation Reference (Template)
# ============================================================================
# This is a reference implementation showing the manual approach to
# summary-based memory (if you wanted to implement it from scratch).
#
# Steps:
# 1. Keep all messages initially
# 2. After threshold, call LLM to summarize old messages
# 3. Replace old messages with summary
# 4. Keep only recent messages + summary
#
# Trade-offs:
# - Most efficient: cheapest token cost
# - Summary may lose granular detail
# - Good for very long conversations
#
# Note: The actual working implementation is in the next cell
# (using LangChain's built-in summarization).
# ============================================================================

# STEP 10: Summary Memory (LLM-based compression)
# store_summary = {}
# keep_recent_messages = 2  # Always keep last 2 turns, summarize the rest
#
# def get_session_history_summary(session_id: str):
#     if session_id not in store_summary:
#         store_summary[session_id] = {
#             "messages": InMemoryChatMessageHistory(),
#             "summary": ""
#         }
#     return store_summary[session_id]["messages"]
#
# def compress_with_llm(messages, llm, keep_recent=2):
#     """Use LLM to summarize old messages"""
#     if len(messages) <= keep_recent * 2:
#         return None  # Not enough messages to summarize
#     
#     old_messages = messages[:-keep_recent*2]
#     old_text = "\n".join([msg.content for msg in old_messages if hasattr(msg, 'content')])
#     
#     summary_prompt = f"""Summarize this customer support conversation in 1-2 sentences, focusing on key issues:
# {old_text}
# Summary:"""
#     
#     summary = llm.invoke(summary_prompt)
#     return summary.content if hasattr(summary, 'content') else str(summary)
#
# qa_chain_summary = RunnableWithMessageHistory(
#     base_chain,
#     get_session_history_summary,
#     input_messages_key="question",
#     history_messages_key="history"
# )
#
# def chat_summary(user_input, session_id="summary_session"):
#     if session_id not in store_summary:
#         get_session_history_summary(session_id)
#     
#     config = {"configurable": {"session_id": session_id}}
#     answer = qa_chain_summary.invoke({"question": user_input}, config=config)
#     
#     history_obj = store_summary[session_id]["messages"]
#     if len(history_obj.messages) > keep_recent_messages * 2:
#         summary = compress_with_llm(history_obj.messages, llm, keep_recent=keep_recent_messages)
#         if summary:
#             store_summary[session_id]["summary"] = summary
#     
#     return answer
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_summary(q)
#     history = store_summary["summary_session"]["messages"].messages
#     summary = store_summary["summary_session"]["summary"]
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     if summary:
#         print(f"Compressed: {summary[:80]}...")
#     print()

print("Step 10: Ready to implement summary memory")


In [ ]:
# ============================================================================
# APPROACH 3: Summary Memory (LLM-Based Compression)
# ============================================================================
# Summary memory uses the LLM to compress old messages into summaries.
#
# Trade-offs:
# ✅ Pros:
#    - Most efficient: lowest token cost for long conversations
#    - Works for unlimited conversation length
#    - Maintains key context via summaries
#
# ❌ Cons:
#    - May lose granular detail in summaries
#    - Extra LLM calls for compression (costs money)
#    - Summaries less precise than original messages
#
# Config:
# - keep_recent_messages = 2: Keep last 2 turns verbatim, summarize older
# - Adjust based on conversation importance
#
# Use case: Research sessions, long support chats, multi-hour Q&A sessions
# ============================================================================

from langchain_community.chat_message_histories import ChatMessageHistory  # For storing messages
from langchain_core.messages import SystemMessage  # For adding summary as system message

store_summary = {}  # Separate store for summary memory
keep_recent_messages = 2  # Keep last 2 turns verbatim, summarize everything older

def get_session_history_summary(session_id: str):
    """Summary memory: stores messages with LLM-generated summaries"""
    if session_id not in store_summary:
        store_summary[session_id] = {
            "messages": InMemoryChatMessageHistory(),  # Stores all messages
            "summary": ""  # Stores running summary
        }
    return store_summary[session_id]["messages"]

def update_summary(session_id: str):
    """Generate summary of old messages and keep only recent ones
    
    Logic:
    1. If messages <= threshold (4), keep all (not enough to summarize)
    2. If messages > threshold:
       a. Extract old messages (everything except last N)
       b. Call LLM to create 1-2 sentence summary
       c. Append to running summary
       d. Keep only recent messages (+ summary stored separately)
    """
    session_data = store_summary[session_id]
    messages = session_data["messages"].messages
    
    # Only summarize if we have more than keep_recent_messages * 2
    if len(messages) <= keep_recent_messages * 2:
        return
    
    # Get old messages to summarize (everything except last N)
    old_messages = messages[:-keep_recent_messages * 2]
    old_text = "\n".join([f"{msg.type}: {msg.content}" for msg in old_messages])
    
    # Use LLM to create summary
    summary_prompt = f"""Summarize this conversation concisely (1-2 sentences):
{old_text}
Summary:"""
    
    summary_response = llm.invoke(summary_prompt)
    new_summary = summary_response.content if hasattr(summary_response, 'content') else str(summary_response)
    
    # Combine old summary with new summary
    if session_data["summary"]:
        session_data["summary"] = f"{session_data['summary']} {new_summary}"
    else:
        session_data["summary"] = new_summary
    
    # Keep only recent messages
    session_data["messages"].messages = messages[-keep_recent_messages * 2:]

# Build chain with summary memory
qa_chain_summary = RunnableWithMessageHistory(
    base_chain,
    get_session_history_summary,  # Use summary-specific history getter
    input_messages_key="question",
    history_messages_key="history"
)

def chat_summary(user_input, session_id="summary_session"):
    """Chat with summary memory - automatically compresses old messages
    
    How it works:
    1. Store all messages initially
    2. After threshold (4 messages), summarize old messages with LLM
    3. Keep only recent messages + running summary
    4. This keeps token usage low while preserving context
    """
    config = {"configurable": {"session_id": session_id}}
    answer = qa_chain_summary.invoke({"question": user_input}, config=config)
    
    # After each turn, check if we need to compress
    update_summary(session_id)
    
    # Get current state for display
    session_data = store_summary[session_id]
    
    return answer, session_data

print("=== SUMMARY MEMORY (LLM-Based Compression) ===\n")

for i, q in enumerate(questions, 1):
    answer, session_data = chat_summary(q)  # Get answer + session data
    
    print(f"Turn {i}: {q}")
    print(f"Bot: {answer}")
    
    # Show current state
    message_count = len(session_data["messages"].messages)
    summary = session_data["summary"]
    
    print(f"Messages in memory: {message_count}")
    if summary:
        print(f"Running summary: {summary[:100]}...")  # Show first 100 chars
    else:
        print("Running summary: (not yet generated)")
    print()


=== SUMMARY MEMORY (LLM-Based Compression) ===

Turn 1: How do I handle reset password via the web app?
Bot: For reset password on the web app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.
Messages in memory: 2
Running summary: (not yet generated)

Turn 2: How do I handle update payment method via the web app?
Bot: For update payment method on the web app, visit Billing > Payments and refresh your card on file. If it persists, open a ticket with logs, region=EU, and a short description. Typical resolution: 24h.
Messages in memory: 4
Running summary: (not yet generated)

Turn 3: How do I handle two-factor auth via the mobile app?
Bot: For two-factor auth on the mobile app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.
Messages in memory: 4
Running summary: The user inquired about

---

## Memory Strategy Comparison

| Strategy | Keep | Token Cost | Context Loss | Best For |
|----------|------|-----------|--------------|----------|
| **Buffer** | All messages | High ↑ | None | Short chats (3-5 turns), need full history |
| **Window** | Last N msgs | Fixed ↓ | Older turns lost | Long chatbots, balance cost & context |
| **Summary** | Recent + compressed | Low ↓↓ | Granular detail lost | Very long chats, efficiency critical |

### When to Choose Each:
- **Buffer**: Classic customer support (short, focused)
- **Window**: Multi-turn chatbots (medium depth, 10-20 turns)
- **Summary**: Research sessions, document Q&A (unlimited turns)

---

## 🚀 Bonus Exercises

**If you finish early:**

1. **Compare memory sizes:** Print out the message count for each strategy at the end. Which uses fewer tokens?

2. **Test with longer conversation:** Add 5-10 more questions. See how buffer memory grows vs window/summary.

3. **Tweak window size:** Try `window_size=2` (only last turn) vs `window_size=8` (last 4 turns). How does context quality change?

4. **Custom summary style:** Modify the summary prompt to focus on specific topics (e.g., billing issues only).

5. **Measure latency:** Time how long each memory strategy takes. Is summary slower due to LLM compression?

6. **Enable LangSmith:** Trace all three strategies in the LangSmith dashboard. Compare token usage!

---

## ✅ Summary

**You've now mastered RAG + Memory in 10 steps!**
- ✅ Built a full RAG pipeline (CSV → chunks → embeddings → retriever → LLM)
- ✅ Implemented 3 memory strategies (Buffer, Window, Summary)
- ✅ Learned cost vs context trade-offs
- ✅ Ready for production customer support chats

**Next level:** Deploy with FastAPI, add multi-user sessions, integrate LangSmith monitoring.

---

## 🤔 Deep Dive: Why No Chunking in This Project?

### Question: Why Don't We Chunk the FAQ Documents?

In this notebook, we load 500 FAQ pairs directly without chunking. Let's explore why this works here—and what changes when you have 500,000 FAQs.

---

## Answer: Why Chunking Isn't Necessary Here

### Scenario: 500 FAQ Pairs (This Project)

| Factor | Why No Chunking? |
|--------|------------------|
| **Document Size** | Each FAQ is naturally small (Q + A ~100-300 words) |
| **Vector DB Load** | 500 documents = minimal memory/search time |
| **Semantic Coherence** | Whole FAQ is one semantic unit; splitting loses context |
| **Retrieval Quality** | Full FAQ → more relevant answers; fragments confuse LLM |
| **Embedding Cost** | 500 embeddings vs 5,000 (chunks)—huge savings |
| **LLM Context Window** | GPT-4o-mini has 128K tokens; 500 FAQs fit easily |

### Example: Why Chunking Hurts Here

```
❌ BAD: Chunking a single FAQ
Q: How do I reset my password via mobile app?
A: Click the login page's "Forgot Password?" link. 
Enter your email. Check your inbox for a reset link.
Click the link and create a new password.

After chunking:
Chunk 1: "Click the login page's "Forgot Password?" link."
Chunk 2: "Enter your email. Check your inbox for a reset link."
Chunk 3: "Click the link and create a new password."

Problem: User asks "reset password"
→ Retriever returns Chunk 1 only (incomplete answer)
→ LLM confused without full context

✅ GOOD: Keep whole FAQ
Document: "Q: How do I reset my password via mobile app?\nA: [full answer]"
→ Retriever returns complete FAQ
→ LLM has all context needed
```

---

## 📈 Scenario 2: 500,000 FAQs—Chunking Strategy Required!

Now imagine your company acquired 500 large FAQ documents (each 10KB+) from competitors. You have **500,000 Q&A pairs**.

### Problem with No Chunking at 500K Scale

| Problem | Impact |
|---------|--------|
| **Memory Explosion** | 500K embeddings × 1536 dims × 4 bytes = 3 GB RAM |
| **Search Latency** | Semantic search through 500K vectors = slow |
| **Irrelevant Retrievals** | Long generic FAQs match too many queries |
| **LLM Context Bloat** | Can't send 500K FAQs to LLM; costs thousands of dollars |
| **Storage Cost** | Chroma DB grows massive; slow disk I/O |

### Recommended Chunking Strategy for 500K FAQs

#### Step 1: Analyze Document Structure

```python
# Categorize FAQs by domain first
FAQ Categories:
- Account & Login (50K FAQs)
- Billing & Payments (75K FAQs)
- Technical Issues (150K FAQs)
- Orders & Shipping (125K FAQs)
- Returns & Refunds (100K FAQs)
```

#### Step 2: Smart Chunking Strategy

**Strategy: Hierarchical Chunking**

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk large FAQs, keep small FAQs whole
def smart_chunk_faq(faq_document, category):
    """
    Logic:
    1. If FAQ is < 500 chars: Keep whole (don't chunk)
    2. If FAQ is 500-2000 chars: Chunk into 2-3 pieces
    3. If FAQ is > 2000 chars: Aggressive chunking (smaller pieces)
    """
    
    content = faq_document['page_content']
    
    if len(content) < 500:
        return [faq_document]  # Keep whole
    
    elif len(content) < 2000:
        chunker = RecursiveCharacterTextSplitter(
            chunk_size=800,      # Larger chunks for medium FAQs
            chunk_overlap=100,   # Keep context between chunks
            separators=["\n\nA:", "\n", " "]
        )
    
    else:  # > 2000 chars
        chunker = RecursiveCharacterTextSplitter(
            chunk_size=500,      # Smaller chunks for large FAQs
            chunk_overlap=150,   # More overlap for coherence
            separators=["\n\nA:", "\nStep", "\n", " "]
        )
    
    chunks = chunker.split_text(content)
    
    # Preserve metadata in each chunk
    chunk_docs = []
    for i, chunk_text in enumerate(chunks):
        from langchain_core.documents import Document
        chunk_docs.append(
            Document(
                page_content=chunk_text,
                metadata={
                    **faq_document.metadata,
                    "chunk_id": i,
                    "total_chunks": len(chunks),
                    "original_source": faq_document.metadata.get('source')
                }
            )
        )
    
    return chunk_docs

# Apply smart chunking
all_chunks = []
for faq_doc in all_500k_faqs:
    chunks = smart_chunk_faq(faq_doc, category=faq_doc.metadata['category'])
    all_chunks.extend(chunks)

print(f"Original docs: 500,000")
print(f"After smart chunking: {len(all_chunks)}")
# Expected result: ~800,000 - 1.2M chunks (50-150% increase, manageable)
```

#### Step 3: Category-Aware Retrieval

```python
# Instead of searching all 1M chunks, narrow down first

category_keywords = {
    "Account": ["login", "password", "account", "email", "2fa", "authentication"],
    "Billing": ["payment", "invoice", "subscription", "card", "charge", "refund"],
    "Technical": ["error", "bug", "crash", "slow", "api", "integration"],
    "Orders": ["order", "shipping", "delivery", "tracking", "cancel"],
    "Returns": ["return", "refund", "exchange", "warranty", "damaged"]
}

def categorized_retriever(user_query, k=5):
    """
    1. Detect user's likely category
    2. Search only in that category's chunks (5-10% of total)
    3. Return top-K most relevant
    """
    # Detect category from query
    query_lower = user_query.lower()
    relevant_category = None
    
    for category, keywords in category_keywords.items():
        if any(kw in query_lower for kw in keywords):
            relevant_category = category
            break
    
    if relevant_category:
        # Filter chunks by category first (huge speed gain)
        category_chunks = [
            chunk for chunk in all_chunks 
            if chunk.metadata.get('category') == relevant_category
        ]
        
        # Search only in 100K-200K chunks instead of 1M
        results = vectorstore.similarity_search(
            user_query,
            k=k,
            filter={"category": relevant_category}  # Metadata filtering
        )
    else:
        # Fallback: search all chunks (expensive)
        results = vectorstore.similarity_search(user_query, k=k)
    
    return results
```

#### Step 4: Performance Comparison

```
NO CHUNKING (500K FAQs):
- Vector store size: 500K embeddings = 3GB
- Search time: ~500-1000ms (slow!)
- Average tokens sent to LLM: 50K (expensive)
- Cost per query: ~$0.50

SMART CHUNKING (1M chunks):
- Vector store size: 1M embeddings = 6GB (acceptable)
- Category pre-filter: Search only 100K chunks
- Search time: ~50-100ms (10x faster)
- Average tokens sent to LLM: 8K (much cheaper)
- Cost per query: ~$0.08

Result: 6x cheaper, 10x faster!
```

---

## Key Takeaways

| Scale | Strategy | Why? |
|-------|----------|------|
| **100-1,000 docs** | No chunking | Each doc is naturally coherent |
| **1,000-10,000 docs** | Optional chunking | Only if docs are large (>5KB each) |
| **10,000+ docs** | Chunking REQUIRED | Memory, cost, and latency demands |
| **100,000+ docs** | Hierarchical chunking | Category pre-filtering essential |

### Decision Tree: Should You Chunk?

```
Is your FAQ dataset < 1,000 documents?
├─ YES: Skip chunking, keep documents whole
└─ NO: Is average FAQ size > 2KB?
        ├─ YES: Use smart chunking (variable size)
        ├─ NO: Keep documents whole
        
Do you have 100K+ documents?
├─ YES: Implement category-aware retrieval
└─ NO: Simple full-text retrieval is fine
```

---

## Next Challenge: Scale This To 500K FAQs

If you wanted to extend this notebook:

1. **Add category metadata** to CSV (which domain each FAQ belongs to)
2. **Implement `smart_chunk_faq()`** function to conditionally chunk
3. **Add metadata filtering** to retriever (only search relevant category)
4. **Benchmark performance** (search latency, cost, token usage)
5. **Monitor Chroma DB** size and query times as you grow

This is the **production-grade RAG architecture** used by companies like Zendesk, Intercom, and Freshdesk!
